# Dry Bean: create and check the Streamlit app

I have completed training, calculated all six evaluation metrics, and saved and reloaded the five model pipelines. I will now use these saved pipelines to build and check my Streamlit app without retraining the models.

I have placed this notebook in the same BITS lab project folder as my original notebook,  test_data.csv ,  model_metrics.csv , and the  model/  folder. The app code is included below, so running the relevant cell will create  app.py .

I will run the numbered cells in order. The first cell checks that the required files are available and displays their location. The app will use only my own saved pipelines and will not train on uploaded data or load model files uploaded by users.

If the saved files are missing after a lab restart, I will locate or restore them before continuing. The outputs displayed in my original notebook are evidence of the earlier run, but they do not contain the saved model files themselves.

## 1. Find the saved files and check the training environment

In [7]:
import importlib.metadata
import importlib.util
import json
import platform
import sys
from pathlib import Path

# Find the project whether Python starts in Assignment 2 or model.
working_dir = Path.cwd().resolve()

for candidate in (working_dir, working_dir.parent):
    if (candidate / "model" / "manifest.json").is_file():
        PROJECT_DIR = candidate
        break
else:
    checked_paths = [
        str(folder / "model" / "manifest.json")
        for folder in (working_dir, working_dir.parent)
    ]
    raise FileNotFoundError(
        "Could not find manifest.json. Checked:\n"
        + "\n".join(checked_paths)
        + "\nConfirm that manifest.json exists inside the model folder."
    )

print("Project folder:", PROJECT_DIR)
print("Python:", platform.python_version())

manifest_path = PROJECT_DIR / "model" / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

required_files = [
    PROJECT_DIR / "test_data.csv",
    PROJECT_DIR / "model_metrics.csv",
]

for filename in manifest["models"].values():
    if "/" in filename or "\\" in filename or not filename.endswith(".joblib"):
        raise ValueError(f"Invalid model filename: {filename}")

    required_files.append(PROJECT_DIR / "model" / filename)

missing_files = [
    str(file) for file in required_files if not file.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "These required files are missing:\n" + "\n".join(missing_files)
    )

inference_packages = ["numpy", "pandas", "scikit-learn", "joblib"]

for package in inference_packages:
    expected_version = manifest["package_versions"][package]
    installed_version = importlib.metadata.version(package)

    if installed_version != expected_version:
        raise RuntimeError(
            f"{package}: training used {expected_version}, "
            f"but this environment has {installed_version}. "
            "Use the original training environment."
        )

    print(f"{package}=={installed_version}")

print(
    "\nSUCCESS: Found the manifest, all five saved pipelines, "
    "test data, and notebook metrics."
)

Project folder: /home/s2026ak05717/Desktop/ML/Assignment 2
Python: 3.12.7
numpy==2.5.4
pandas==2.2.3
scikit-learn==1.9.1
joblib==1.4.2

SUCCESS: Found the manifest, all five saved pipelines, test data, and notebook metrics.


## 2. Create app.py without discarding an existing app

In [10]:
APP_SOURCE = 'import csv\nimport io\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    matthews_corrcoef,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.utils.validation import check_is_fitted\n\n\nMODEL_NAMES = (\n    "Logistic Regression",\n    "Decision Tree",\n    "KNN",\n    "Naive Bayes",\n    "Support Vector Machine",\n)\nMETRIC_COLUMNS = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]\n\n\ndef read_manifest(project_dir):\n    manifest_path = Path(project_dir) / "model" / "manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    features = manifest["features"]\n    classes = manifest["classes"]\n    models = manifest["models"]\n    if (\n        not isinstance(features, list)\n        or len(features) != 16\n        or not all(isinstance(value, str) and value for value in features)\n        or len(set(features)) != 16\n    ):\n        raise ValueError("The manifest must contain 16 distinct feature names.")\n    if (\n        not isinstance(classes, list)\n        or len(classes) != 7\n        or not all(isinstance(value, str) and value for value in classes)\n        or len(set(classes)) != 7\n    ):\n        raise ValueError("The manifest must contain seven distinct class labels.")\n    if manifest["target"] != "Class" or "Class" in features:\n        raise ValueError("The manifest target must be Class, separate from the features.")\n    if not isinstance(models, dict) or set(models) != set(MODEL_NAMES):\n        raise ValueError("The manifest must identify all five required models.")\n    for filename in models.values():\n        if (\n            not isinstance(filename, str)\n            or "/" in filename\n            or "\\\\" in filename\n            or Path(filename).suffix != ".joblib"\n        ):\n            raise ValueError("Model filenames must be repository-local .joblib files.")\n    return manifest\n\n\ndef read_test_csv(payload, manifest):\n    if not payload or not payload.strip():\n        raise ValueError("The uploaded CSV is empty.")\n    try:\n        text = payload.decode("utf-8-sig")\n        reader = csv.reader(io.StringIO(text, newline=""), strict=True)\n        header = next(reader, [])\n        if not header:\n            raise ValueError("The uploaded CSV has no column header.")\n        if len(set(header)) != len(header):\n            raise ValueError("Duplicate column names are not allowed.")\n        required = manifest["features"] + [manifest["target"]]\n        missing = [column for column in required if column not in header]\n        extra = [column for column in header if column not in required]\n        if missing or extra:\n            details = []\n            if missing:\n                details.append("Missing required columns: " + ", ".join(missing))\n            if extra:\n                details.append("Unexpected columns: " + ", ".join(extra))\n            raise ValueError(". ".join(details))\n        for row_number, row in enumerate(reader, start=2):\n            if row and len(row) != len(header):\n                raise ValueError(\n                    f"CSV row {row_number} has {len(row)} fields; "\n                    f"expected {len(header)}."\n                )\n        frame = pd.read_csv(\n            io.StringIO(text), dtype=str, keep_default_na=False, on_bad_lines="error"\n        )\n    except (UnicodeError, csv.Error, pd.errors.ParserError, pd.errors.EmptyDataError) as error:\n        raise ValueError(f"Cannot read the CSV: {error}") from error\n    if frame.empty:\n        raise ValueError("The uploaded CSV contains no test rows.")\n    features = frame.loc[:, manifest["features"]].copy()\n    for column in manifest["features"]:\n        try:\n            features[column] = features[column].astype(float)\n        except (ValueError, TypeError) as error:\n            raise ValueError(\n                f"Invalid numerical values in {column}; all feature values must be numeric."\n            ) from error\n        if not np.isfinite(features[column].to_numpy()).all():\n            raise ValueError(\n                f"Invalid numerical values in {column}; missing values and infinity "\n                "are not allowed."\n            )\n    labels = frame[manifest["target"]]\n    unknown = sorted(set(labels) - set(manifest["classes"]))\n    if unknown:\n        raise ValueError("Unknown class labels: " + ", ".join(repr(value) for value in unknown))\n    return features, labels\n\n\ndef load_pipeline(project_dir, name, manifest):\n    if name not in MODEL_NAMES:\n        raise ValueError("Unknown model selection.")\n    artifact = Path(project_dir) / "model" / manifest["models"][name]\n    # Only repository artifacts are loaded; uploaded files are always CSV data.\n    pipeline = joblib.load(artifact)\n    if not isinstance(pipeline, Pipeline):\n        raise TypeError(f"{name} must be a complete fitted sklearn Pipeline.")\n    check_is_fitted(pipeline)\n    if not callable(getattr(pipeline, "predict_proba", None)):\n        raise TypeError(f"{name} must support predict_proba for multiclass AUC.")\n    if set(pipeline.classes_) != set(manifest["classes"]):\n        raise ValueError(f"{name} class labels do not match the manifest.")\n    if list(pipeline.feature_names_in_) != manifest["features"]:\n        raise ValueError(f"{name} feature names do not match the manifest.")\n    return pipeline\n\n\ndef evaluate_pipeline(pipeline, features, actual_labels, classes):\n    predicted_labels = np.asarray(pipeline.predict(features))\n    probabilities = np.asarray(pipeline.predict_proba(features))\n    model_labels = np.asarray(pipeline.classes_)\n    actual_labels = np.asarray(actual_labels)\n    if len(actual_labels) == 0:\n        raise ValueError("At least one test row is required.")\n    if set(model_labels) != set(classes):\n        raise ValueError("The model must be trained on all seven manifest classes.")\n    if not set(actual_labels).issubset(set(model_labels)):\n        raise ValueError("The test data contains an unknown bean class.")\n    if predicted_labels.shape != actual_labels.shape:\n        raise ValueError("Prediction dimensions do not match the test labels.")\n    if not set(predicted_labels).issubset(set(model_labels)):\n        raise ValueError("The model returned an unknown class label.")\n    if probabilities.shape != (len(actual_labels), len(model_labels)):\n        raise ValueError("Probability dimensions do not match the test data.")\n    if not np.isfinite(probabilities).all():\n        raise ValueError("Predicted probabilities contain missing or infinite values.")\n    if (probabilities < 0).any() or (probabilities > 1).any():\n        raise ValueError("Predicted probabilities must be between 0 and 1.")\n    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6):\n        raise ValueError("Class probabilities for each row must sum to 1.")\n    if set(actual_labels) == set(model_labels):\n        auc = roc_auc_score(\n            actual_labels, probabilities, labels=model_labels,\n            multi_class="ovr", average="macro",\n        )\n        auc_note = "Macro one-vs-rest AUC calculated for all seven classes."\n    else:\n        auc = np.nan\n        auc_note = (\n            "Full multiclass AUC unavailable: the test data does not contain "\n            "all seven classes."\n        )\n    averaging = {"labels": classes, "average": "macro", "zero_division": 0}\n    metrics = {\n        "Accuracy": accuracy_score(actual_labels, predicted_labels),\n        "AUC": auc,\n        "Precision": precision_score(actual_labels, predicted_labels, **averaging),\n        "Recall": recall_score(actual_labels, predicted_labels, **averaging),\n        "F1": f1_score(actual_labels, predicted_labels, **averaging),\n        "MCC": matthews_corrcoef(actual_labels, predicted_labels),\n    }\n    return metrics, predicted_labels, auc_note\n\n\ndef main():\n    import streamlit as st\n\n    st.set_page_config(page_title="Dry Bean Model Explorer", layout="wide")\n    st.title("Dry Bean Model Explorer")\n    st.caption("Evaluate saved pipelines on held-out CSV data. No training occurs in this app.")\n    project_dir = Path(__file__).resolve().parent\n    try:\n        manifest = read_manifest(project_dir)\n    except (OSError, ValueError, TypeError, KeyError) as error:\n        st.error(f"Cannot read model/manifest.json: {error}")\n        st.stop()\n\n    with st.sidebar:\n        st.header("Test your models")\n        uploaded = st.file_uploader("Upload held-out test data", type=["csv"])\n        selected = st.selectbox("Saved model", list(MODEL_NAMES))\n        compare = st.checkbox("Compare all five models on these rows")\n        st.caption("CSV only. Never upload joblib or pickle files from an untrusted source.")\n    st.info(\n        "Use test_data.csv or other genuinely held-out data. The CSV must contain "\n        "the exact 16 feature columns and Class. Column order may differ."\n    )\n    st.caption(\n        "Accuracy: fraction correct. Precision/Recall/F1: macro average over all "\n        "seven manifest classes, zero_division=0. AUC: macro one-vs-rest from "\n        "class probabilities. MCC: multiclass Matthews correlation coefficient."\n    )\n    if uploaded is None:\n        st.write("Upload your test CSV using the sidebar to begin.")\n        with st.expander("Required CSV columns"):\n            st.code(", ".join(manifest["features"] + [manifest["target"]]))\n        return\n    try:\n        features, actual = read_test_csv(uploaded.getvalue(), manifest)\n    except (ValueError, TypeError, KeyError) as error:\n        st.error(f"CSV validation failed: {error}")\n        st.stop()\n    st.success(f"Validated {len(features):,} test rows and 16 features.")\n    with st.expander("Preview validated data"):\n        preview = features.copy()\n        preview[manifest["target"]] = actual\n        st.dataframe(preview.head(50), hide_index=True)\n\n    try:\n        pipeline = load_pipeline(project_dir, selected, manifest)\n        metrics, predicted, auc_note = evaluate_pipeline(\n            pipeline, features, actual, manifest["classes"]\n        )\n    except (OSError, ValueError, TypeError, KeyError, AttributeError, ImportError) as error:\n        st.error(f"Cannot evaluate {selected}: {error}")\n        st.stop()\n    st.subheader(selected)\n    for offset in (0, 3):\n        for column, metric in zip(st.columns(3), METRIC_COLUMNS[offset:offset + 3]):\n            value = metrics[metric]\n            column.metric(metric, "Unavailable" if pd.isna(value) else f"{value:.4f}")\n    if pd.isna(metrics["AUC"]):\n        st.warning(auc_note)\n    else:\n        st.caption(auc_note)\n    confusion_tab, report_tab, prediction_tab = st.tabs(\n        ["Confusion matrix", "Classification report", "Predictions"]\n    )\n    with confusion_tab:\n        counts = confusion_matrix(actual, predicted, labels=manifest["classes"])\n        st.caption("Rows: actual class. Columns: predicted class.")\n        st.dataframe(pd.DataFrame(\n            counts,\n            index=pd.Index(manifest["classes"], name="Actual"),\n            columns=pd.Index(manifest["classes"], name="Predicted"),\n        ))\n    with report_tab:\n        report = classification_report(\n            actual, predicted, labels=manifest["classes"],\n            output_dict=True, zero_division=0,\n        )\n        st.dataframe(pd.DataFrame(report).transpose().round(4))\n    with prediction_tab:\n        results = features.copy()\n        results["Actual Class"] = actual.to_numpy()\n        results["Predicted Class"] = predicted\n        st.dataframe(results, hide_index=True)\n        st.download_button(\n            "Download predictions", results.to_csv(index=False).encode("utf-8"),\n            file_name="dry_bean_predictions.csv", mime="text/csv",\n        )\n    if compare:\n        rows = []\n        try:\n            with st.spinner("Evaluating all five saved pipelines on the same rows..."):\n                for name in MODEL_NAMES:\n                    if name == selected:\n                        model_metrics = metrics\n                    else:\n                        other = load_pipeline(project_dir, name, manifest)\n                        model_metrics, _, _ = evaluate_pipeline(\n                            other, features, actual, manifest["classes"]\n                        )\n                    rows.append({"ML Model Name": name, **model_metrics})\n        except (OSError, ValueError, TypeError, KeyError, AttributeError, ImportError) as error:\n            st.error(f"Model comparison failed for {name}: {error}")\n            st.stop()\n        table = pd.DataFrame(rows).set_index("ML Model Name")\n        st.subheader("Same-data model comparison")\n        st.dataframe(table[METRIC_COLUMNS].round(4))\n        st.caption("AUC is shown as missing when a class is absent; the other metrics remain valid.")\n        st.download_button(\n            "Download comparison", table.to_csv().encode("utf-8"),\n            file_name="uploaded_data_metrics.csv", mime="text/csv",\n        )\n\n\nif __name__ == "__main__":\n    main()\n'

app_path = PROJECT_DIR / "app.py"
if app_path.exists() and app_path.read_text(encoding="utf-8") != APP_SOURCE:
    backup_path = PROJECT_DIR / "app.py.before_streamlit_setup.bak"
    if backup_path.exists():
        raise FileExistsError(
            f"An app and an earlier backup already exist. Review {app_path} "
            f"and {backup_path} before replacing either file."
        )
    backup_path.write_bytes(app_path.read_bytes())
    print("Preserved your previous app:", backup_path)
app_path.write_text(APP_SOURCE, encoding="utf-8")

constraints_path = PROJECT_DIR / "training_constraints.txt"
constraint_lines = [
    f"{package}=={manifest['package_versions'][package]}"
    for package in inference_packages
]
constraints_path.write_text("\n".join(constraint_lines) + "\n", encoding="utf-8")
print("Created:", app_path)
print("Created:", constraints_path)


Created: /home/s2026ak05717/Desktop/ML/Assignment 2/app.py
Created: /home/s2026ak05717/Desktop/ML/Assignment 2/training_constraints.txt


## 3. Install Streamlit

The constraints protect the four inference packages used during training.

In [13]:
import subprocess

if importlib.util.find_spec("streamlit") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--constraint", str(constraints_path), "streamlit",
    ])
else:
    print("Streamlit is already installed; no installation needed.")

for package in inference_packages:
    actual = importlib.metadata.version(package)
    if actual != manifest["package_versions"][package]:
        raise RuntimeError(f"{package} changed unexpectedly; restore the training environment.")

requirements = [
    f"{package}=={importlib.metadata.version(package)}"
    for package in inference_packages + ["streamlit"]
]
requirements_path = PROJECT_DIR / "requirements.txt"
requirements_text = "\n".join(requirements) + "\n"
if requirements_path.exists() and requirements_path.read_text(encoding="utf-8") != requirements_text:
    backup = PROJECT_DIR / "requirements.txt.before_streamlit_setup.bak"
    if backup.exists():
        raise FileExistsError(f"Review the existing requirements and backup first: {backup}")
    backup.write_bytes(requirements_path.read_bytes())
requirements_path.write_text(requirements_text, encoding="utf-8")
print("Created requirements.txt using the actual lab versions:")
print(requirements_text)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 166.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 122.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.36.2 which is incompatible.


Created requirements.txt using the actual lab versions:
numpy==2.5.4
pandas==2.2.3
scikit-learn==1.9.1
joblib==1.4.2
streamlit==1.66.1



## 4. Compare the app's six metrics against the notebook

This loads your own saved pipelines and uses test_data.csv. It does not train.

In [16]:
import numpy as np
import pandas as pd

spec = importlib.util.spec_from_file_location("drybean_app", PROJECT_DIR / "app.py")
drybean_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(drybean_app)
checked_manifest = drybean_app.read_manifest(PROJECT_DIR)
payload = (PROJECT_DIR / "test_data.csv").read_bytes()
features, actual_labels = drybean_app.read_test_csv(payload, checked_manifest)
if set(actual_labels) != set(checked_manifest["classes"]):
    raise ValueError("The original test_data.csv must contain all seven classes for this check.")
expected = pd.read_csv(PROJECT_DIR / "model_metrics.csv", index_col=0)

rows = []
for name in drybean_app.MODEL_NAMES:
    pipeline = drybean_app.load_pipeline(PROJECT_DIR, name, checked_manifest)
    metrics, _, _ = drybean_app.evaluate_pipeline(
        pipeline, features, actual_labels, checked_manifest["classes"]
    )
    np.testing.assert_allclose(
        [metrics[column] for column in drybean_app.METRIC_COLUMNS],
        expected.loc[name, drybean_app.METRIC_COLUMNS].to_numpy(dtype=float),
        rtol=0, atol=1e-12, equal_nan=False,
        err_msg=f"{name}: the app does not match the saved notebook metrics.",
    )
    rows.append({"ML Model Name": name, **metrics})
    print(f"PASS: {name} - all six metrics match the notebook.")

app_metrics = pd.DataFrame(rows).set_index("ML Model Name")
display(app_metrics.round(4))

test_frame = features.copy()
test_frame[checked_manifest["target"]] = actual_labels.to_numpy()
reordered = test_frame.loc[:, list(reversed(test_frame.columns))]
reordered_features, reordered_labels = drybean_app.read_test_csv(
    reordered.to_csv(index=False).encode("utf-8"), checked_manifest
)
np.testing.assert_array_equal(reordered_features.to_numpy(), features.to_numpy())
np.testing.assert_array_equal(reordered_labels.to_numpy(), actual_labels.to_numpy())
print("PASS: feature columns are reordered using the manifest.")

bad_inputs = {
    "empty CSV": b"",
    "header only": test_frame.iloc[:0].to_csv(index=False).encode("utf-8"),
    "missing column": test_frame.drop(columns=[features.columns[0]]).to_csv(index=False).encode("utf-8"),
    "extra column": test_frame.assign(unexpected=1).to_csv(index=False).encode("utf-8"),
}
bad_numeric = test_frame.copy()
bad_numeric[features.columns[0]] = bad_numeric[features.columns[0]].astype(str)
bad_numeric.loc[bad_numeric.index[0], features.columns[0]] = "not numeric"
bad_inputs["invalid number"] = bad_numeric.to_csv(index=False).encode("utf-8")
bad_label = test_frame.copy()
bad_label.loc[bad_label.index[0], checked_manifest["target"]] = "UNKNOWN"
bad_inputs["unknown class"] = bad_label.to_csv(index=False).encode("utf-8")
for description, invalid_payload in bad_inputs.items():
    try:
        drybean_app.read_test_csv(invalid_payload, checked_manifest)
    except ValueError as error:
        print(f"PASS: rejected {description}: {error}")
    else:
        raise AssertionError(f"Invalid input was accepted: {description}")

subset = test_frame.loc[
    test_frame[checked_manifest["target"]] != checked_manifest["classes"][0]
]
subset_features, subset_labels = drybean_app.read_test_csv(
    subset.to_csv(index=False).encode("utf-8"), checked_manifest
)
subset_metrics, _, note = drybean_app.evaluate_pipeline(
    pipeline, subset_features, subset_labels, checked_manifest["classes"]
)
assert np.isnan(subset_metrics["AUC"]), "AUC must be unavailable when a class is missing."
assert all(np.isfinite(value) for key, value in subset_metrics.items() if key != "AUC")
print("PASS:", note, "The other five metrics remain available.")
print("\nApp file and data checks passed. Next, start Streamlit.")


PASS: Logistic Regression - all six metrics match the notebook.
PASS: Decision Tree - all six metrics match the notebook.
PASS: KNN - all six metrics match the notebook.
PASS: Naive Bayes - all six metrics match the notebook.
PASS: Support Vector Machine - all six metrics match the notebook.


,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.9192,0.9947,0.9307,0.9300,0.9302,0.9023
Decision Tree,0.9011,0.9703,0.9160,0.9152,0.9155,0.8804
KNN,0.9136,0.9854,0.9268,0.9237,0.9250,0.8956
Naive Bayes,0.7630,0.9670,0.7736,0.7685,0.7669,0.7143
Support Vector Machine,0.9210,0.9945,0.9342,0.9317,0.9328,0.9045


PASS: feature columns are reordered using the manifest.
PASS: rejected empty CSV: The uploaded CSV is empty.
PASS: rejected header only: The uploaded CSV contains no test rows.
PASS: rejected missing column: Missing required columns: Area
PASS: rejected extra column: Unexpected columns: unexpected
PASS: rejected invalid number: Invalid numerical values in Area; all feature values must be numeric.
PASS: rejected unknown class: Unknown class labels: 'UNKNOWN'
PASS: Full multiclass AUC unavailable: the test data does not contain all seven classes. The other five metrics remain available.

App file and data checks passed. Next, start Streamlit.


## 5. Start Streamlit 

In [24]:
import socket
import subprocess
import sys
import time
from urllib.error import URLError
from urllib.request import urlopen

PORT = 8501

if "streamlit_process" in globals() and streamlit_process.poll() is None:
    raise RuntimeError(
        f"My Streamlit app is already running with PID {streamlit_process.pid}. "
        "I need to use the existing app or stop it before starting it again."
    )

with socket.socket() as probe:
    try:
        probe.bind(("127.0.0.1", PORT))
    except OSError as error:
        raise RuntimeError(
            f"Port {PORT} is already in use. "
            "I need to choose another port without stopping another user's process."
        ) from error

log_path = PROJECT_DIR / "streamlit.log"

with log_path.open("a", encoding="utf-8") as log:
    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(PROJECT_DIR / "app.py"),
            "--server.address=127.0.0.1",
            f"--server.port={PORT}",
            "--server.headless=true",
            "--browser.gatherUsageStats=false",
        ],
        cwd=str(PROJECT_DIR),
        stdout=log,
        stderr=subprocess.STDOUT,
    )

healthy = False

for _ in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError(
            "My Streamlit app stopped before becoming ready. "
            "The latest log messages are:\n"
            + log_path.read_text(encoding="utf-8")[-4000:]
        )

    try:
        with urlopen(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=1,
        ) as response:
            healthy = (
                response.status == 200
                and response.read().strip() == b"ok"
            )

        if healthy:
            break

    except (URLError, TimeoutError, OSError):
        time.sleep(0.5)

if not healthy:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
    raise RuntimeError(
        f"My Streamlit app did not become ready. "
        f"I need to check the log file: {log_path}"
    )

print("My Streamlit server is running in the BITS VM.")
print("Process ID:", streamlit_process.pid)
print(f"VM-local address: http://127.0.0.1:{PORT}")
print(
    f"I can access it through the VM browser or the lab's approved "
    f"port preview/forwarding for port {PORT}, if available."
)
print("This address refers to the BITS VM, not my own computer.")
print("I will keep this notebook kernel running while using the app.")
print("Log file:", log_path)


RuntimeError: My Streamlit app is already running with PID 15642. I need to use the existing app or stop it before starting it again.

In [26]:

from urllib.request import urlopen

with urlopen(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=5,
) as response:
    print("HTTP status:", response.status)
    print("Streamlit health:", response.read().decode().strip())


HTTP status: 200
Streamlit health: ok
